# Developer Experience

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LakeLogic/LakeLogic/blob/main/examples/colab/04_developer_experience.ipynb)

Structured diagnostics, DDL generation, DAG visualization, dry run previews, surgical resets, and multi-channel alerts — all from the contract.

In [9]:
import os
import subprocess
import sys

# ── Install lakelogic ─────────────────────────────────────────────────
# Update the path below to match your local lakelogic checkout.
# On Colab (or if the path doesn't exist), falls back to PyPI.
_LAKELOGIC_LOCAL = r"C:\_Personal\_SaaS\lakelogic"

if os.path.isdir(_LAKELOGIC_LOCAL):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", _LAKELOGIC_LOCAL, "-q"])
    print(f"\u2705 Installed lakelogic (editable) from {_LAKELOGIC_LOCAL}")
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "lakelogic", "-q"])
    print("\u2705 Installed lakelogic from PyPI")

✅ Installed lakelogic (editable) from C:\_Personal\_SaaS\lakelogic


In [10]:
import subprocess
import sys
import importlib
import urllib.request
import os

if importlib.util.find_spec("lakelogic") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lakelogic[polars]"])
if not os.path.exists("_setup.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/LakeLogic/LakeLogic/main/examples/colab/_setup.py", "_setup.py"
    )
from _setup import *

---
## 1. Structured Diagnostics — Powered by `loguru`

**The Problem:** Your pipeline failed. The log says `ERROR: validation failed`. No contract name, no run ID, no timestamp with timezone. You grep through 50 log files.

**The Solution:** LakeLogic uses `loguru` for structured logging. Every line includes precise timestamps, severity levels, the exact function path, and execution tags — drastically cutting troubleshooting time.

In [11]:
# LakeLogic uses loguru out of the box — no logging config needed.
# Just run a processor and observe the structured log output.

contract = write_contract(
    """
version: 1.0.0
dataset: diagnostics_demo
model:
  fields:
    - name: id
      type: integer
      required: true
    - name: value
      type: string
quality:
  row_rules:
    - name: has_value
      sql: "value IS NOT NULL AND value != ''"
""",
    "diag.yaml",
)

proc = DataProcessor(contract, engine="polars")
source_df = DataGenerator(contract).generate(rows=50, invalid_ratio=0.1)
good, bad = proc.run(source_df)

# ── Observe: every log line has timestamp | level | module:function:line ──
print(f"\nResult: good={len(good)}, bad={len(bad)}")
print("\n\u2705 Every log line above includes:")
print("   • ISO timestamp with timezone")
print("   • Severity level (INFO, WARNING, ERROR)")
print("   • Module path: lakelogic.core.processor:run:788")
print("   • Run metrics: Source count, Good/Quarantine split, ratio")

2026-04-14 06:13:01.421 | INFO     | lakelogic.core.generator:generate:3083 - 📋 Generating data for: diagnostics_demo
2026-04-14 06:13:01.421 | INFO     | lakelogic.core.generator:generate:3084 -    Records    : 45 valid + 5 invalid = 50 total
2026-04-14 06:13:01.421 | INFO     | lakelogic.core.generator:generate:3102 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-04-14 06:13:01.423 | INFO     | lakelogic.core.generator:generate:3117 -    Edge cases : Heuristic-only (no AI edge cases available)
2026-04-14 06:13:01.424 | INFO     | lakelogic.core.generator:generate:3151 -    Row generation complete: 50 records built
2026-04-14 06:13:01.424 | INFO     | lakelogic.core.generator:generate:3173 -    Test cases : 6 across 3 categories
2026-04-14 06:13:01.424 | INFO     | lakelogic.core.generator:generate:3175 -      EMPTY_STRING                      3 injections
2026-04-14 06:13:01.427 | INFO     | lakelogic.core.generator:generate:3175 -      NOT_NULL_VIOLATION   


Result: good=44, bad=6

✅ Every log line above includes:
   • ISO timestamp with timezone
   • Severity level (INFO, WARNING, ERROR)
   • Module path: lakelogic.core.processor:run:788
   • Run metrics: Source count, Good/Quarantine split, ratio


---
## 2. DDL-Only Mode — Schema Migration Without Running a Pipeline

**The Problem:** You need to create the target table schema before the first pipeline run, but you don't want to process any data yet.

**The Solution:** `DataProcessor.generate_ddl()` generates CREATE TABLE DDL directly from the contract — perfect for CI/CD migrations.

In [12]:
# ── Write a contract with a rich schema ──────────────────────────────
ddl_contract = write_contract(
    """
version: 1.0.0
dataset: user_events
model:
  fields:
    - name: event_id
      type: integer
      required: true
    - name: user_id
      type: string
      required: true
    - name: event_type
      type: string
    - name: payload
      type: string
    - name: created_at
      type: string
      required: true
""",
    "user_events.yaml",
)

# ── Generate DDL for different backends ──────────────────────────────
proc = DataProcessor(ddl_contract, engine="duckdb")

print("DuckDB DDL:")
print(proc.generate_ddl(backend="duckdb"))

print("\nSpark DDL:")
print(proc.generate_ddl(backend="spark"))

print("\n\u2705 Schema migration from contract — no pipeline run needed.")

DuckDB DDL:
CREATE TABLE IF NOT EXISTS user_events (
  event_id INTEGER NOT NULL,
  user_id VARCHAR NOT NULL,
  event_type VARCHAR,
  payload VARCHAR,
  created_at VARCHAR NOT NULL
);

Spark DDL:
CREATE TABLE IF NOT EXISTS user_events (
  event_id INT NOT NULL,
  user_id STRING NOT NULL,
  event_type STRING,
  payload STRING,
  created_at STRING NOT NULL
)
USING DELTA
TBLPROPERTIES ('delta.enableDeletionVectors' = false);

✅ Schema migration from contract — no pipeline run needed.


---
## 3. DAG Dependency Viewer — Execution Order at a Glance

**The Problem:** You have 15 contracts with dependencies. Running them in the wrong order corrupts downstream tables.

**The Solution:** `LakehousePipeline.visualize_dag()` renders the full dependency graph from your `_system.yaml` registry — showing bronze → silver → gold flow and execution order.

In [13]:
import os
import yaml
from lakelogic.core.registry import DomainRegistry
from lakelogic.pipeline import LakehousePipeline
from IPython.display import HTML, display

# ── Create inline contracts ──────────────────────────────────────────
DAG_DIR = "./dag_demo"
os.makedirs(f"{DAG_DIR}/contracts/bronze", exist_ok=True)
os.makedirs(f"{DAG_DIR}/contracts/silver", exist_ok=True)

# Bronze: orders (raw landing)
write_contract(
    """
version: 1.0.0
dataset: orders
info:
  title: bronze_demo_orders
  target_layer: bronze
source:
  type: landing
  path: "./dag_demo/landing/orders"
  format: ndjson
model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: amount
      type: float
""",
    f"{DAG_DIR}/contracts/bronze/bronze_demo_orders_v1.0.yaml",
)

# Bronze: customers (raw landing)
write_contract(
    """
version: 1.0.0
dataset: customers
info:
  title: bronze_demo_customers
  target_layer: bronze
source:
  type: landing
  path: "./dag_demo/landing/customers"
  format: ndjson
model:
  fields:
    - name: customer_id
      type: integer
      required: true
    - name: name
      type: string
      pii: true
""",
    f"{DAG_DIR}/contracts/bronze/bronze_demo_customers_v1.0.yaml",
)

# Silver: orders_cleaned (depends on bronze orders + customers)
write_contract(
    """
version: 1.0.0
dataset: orders_cleaned
info:
  title: silver_demo_orders_cleaned
  target_layer: silver
source:
  type: table
  path: "./dag_demo/lakehouse/bronze/bronze_demo_orders"
model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: amount
      type: float

downstream:
  - type: dashboard
    name: "Weekly Sales Performance"
    platform: power_bi
    url: "https://app.powerbi.com/..."
    owner: "marketing-analytics"
    
""",
    f"{DAG_DIR}/contracts/silver/silver_demo_orders_v1.0.yaml",
)

print("\u2705 Inline contracts created")

# ── Create inline _system.yaml registry ──────────────────────────────
system_yaml = {
    "domain": "demo",
    "system": "ecommerce",
    "external_sources": [
        {
            "name": "ecommerce demo API",
            "source_domain": "ecommerce Vendor",
            "catalog_path": "external_storage_path_or_api",
            "consumed_by": ["orders", "customers"],
        }
    ],
    "contracts": [
        {
            "layer": "bronze",
            "entity": "orders",
            "path": "contracts/bronze/bronze_demo_orders_v1.0.yaml",
            "depends_on": ["customers"],
            "enabled": True,
        },
        {
            "layer": "bronze",
            "entity": "customers",
            "path": "contracts/bronze/bronze_demo_customers_v1.0.yaml",
            "enabled": True,
        },
        {
            "layer": "silver",
            "entity": "orders_cleaned",
            "path": "contracts/silver/silver_demo_orders_v1.0.yaml",
            "depends_on": ["orders", "customers"],
            "enabled": True,
        },
    ],
    "environments": {
        "local": {
            "catalog": "local",
            "storage_root": "./dag_demo/lakehouse",
            "data_root": "./dag_demo/lakehouse",
            "quarantine_root": "./dag_demo/lakehouse/_quarantine",
        }
    },
    "storage": {
        "external_location_root": "./dag_demo/lakehouse",
    },
}

system_path = f"{DAG_DIR}/_system.yaml"
with open(system_path, "w") as f:
    yaml.dump(system_yaml, f, default_flow_style=False)
print(f"\u2705 _system.yaml written to {system_path}")

# ── Build pipeline and visualize ─────────────────────────────────────
registry = DomainRegistry.from_yaml(system_path, environment="local", storage_mode="direct")
pipeline = LakehousePipeline(registry, engine="polars")

display(HTML(pipeline.visualize_dag()))

✅ Inline contracts created
✅ _system.yaml written to ./dag_demo/_system.yaml


---
## 4. Dry Run Mode — Preview Before You Commit

**The Problem:** You changed a transformation and want to see the execution plan before it touches production data.

**The Solution:** Run with `dry_run=True`. The pipeline walks every contract in topological order and logs what it **would** execute — but **skips all processing and writes**.

In [14]:
# ── Dry Run: uses the same inline pipeline from Section 3 ───────────
summary = pipeline.run(
    target_layers="bronze,silver",
    reset_layers="",
    reload_layers="",
    dry_run=True,  # <--- PREVIEW ONLY
    entity_filter="",
    environment="local",
    parallel=True,
    max_workers=4,
    ddl_only=False,
    created_by="developer_experience_demo",
    reprocess_from=None,
    reprocess_to=None,
    reprocess_column=None,
    reprocess_values=None,
    retry_attempts=3,
    retry_base_wait_seconds=2,
    entity_timeout_minutes=60,
    max_consecutive_failures=2,
)

print("\n" + "=" * 60)
print("DRY RUN SUMMARY")
print("=" * 60)
print(f"  Pipeline run  : {summary.run_id}")
print(f"  Environment   : {summary.environment}")
print(f"  Dry run       : {summary.dry_run}")
print()
for r in summary.results:
    contract = r.get("contract", "?")
    layer = r.get("layer", "?")
    status = r.get("status", "?")
    print(f"  [{layer:6s}] {contract:25s} -> {status}")
print("=" * 60)
print("\n✅ Every contract was evaluated but nothing was processed or written.")

2026-04-14 06:56:57.772 | INFO     | lakelogic.pipeline.runner:run:1399 - Pipeline storage mode: direct


AttributeError: 'NoneType' object has no attribute 'split'

---
## 5. Surgical Reset & Reload — Silver Only, Bronze Untouched

**The Problem:** Your Silver transformation logic had a bug. You need to re-run Silver without reprocessing Bronze (which took 4 hours).

**The Solution:** Reset and reload just the Silver layer. Bronze output stays untouched — simply re-feed Bronze data into the fixed Silver processor.

In [7]:
# ── Simulate a Bronze → Silver pipeline using LakehousePipeline ─────────

# Generate source data first so the pipeline can run
import polars as pl

os.makedirs("./dag_demo/landing/orders", exist_ok=True)
pl.DataFrame({"order_id": [1, 2], "amount": [10.5, 20.0]}).write_ndjson("./dag_demo/landing/orders/data.ndjson")

os.makedirs("./dag_demo/landing/customers", exist_ok=True)
pl.DataFrame({"customer_id": [1, 2], "name": ["Alice", "Bob"]}).write_ndjson("./dag_demo/landing/customers/data.ndjson")

# RUN 1: Full Pipeline Execute (Bronze + Silver)
print("── RUN 1: Full Pipeline (Bronze + Silver) ──")
summary_1 = pipeline.run(
    target_layers="bronze,silver",
    reset_layers="",
    reload_layers="",
    dry_run=False,
    entity_filter="",
    environment="local",
    parallel=True,
    max_workers=4,
    ddl_only=False,
    created_by="developer_experience_demo",
    reprocess_from=None,
    reprocess_to=None,
    reprocess_column=None,
    reprocess_values=None,
    retry_attempts=3,
    retry_base_wait_seconds=2,
    entity_timeout_minutes=60,
    max_consecutive_failures=2,
)
print(f"Run 1 completed. Processed: {[r['contract'] for r in summary_1.results if r['status'] == 'success']}")
print()

# RUN 2: Surgical Reset & Reload (Silver Only)
# In production you'd fix a transformation bug, then reload Silver natively.
print("── RUN 2: Fix bug, Reset & Reload Silver Only ──")
summary_2 = pipeline.run(
    target_layers="silver",  # <--- TARGET ONLY SILVER
    reset_layers="silver",  # <--- WIPE SILVER TARGETS
    reload_layers="silver",  # <--- IGNORE WATERMARKS FOR SILVER
    dry_run=False,
    entity_filter="",
    environment="local",
    parallel=True,
    max_workers=4,
    ddl_only=False,
    created_by="developer_experience_demo",
    reprocess_from=None,
    reprocess_to=None,
    reprocess_column=None,
    reprocess_values=None,
    retry_attempts=3,
    retry_base_wait_seconds=2,
    entity_timeout_minutes=60,
    max_consecutive_failures=2,
)
print(f"Run 2 completed. Processed: {[r['contract'] for r in summary_2.results if r['status'] == 'success']}")

print("\n✅ Silver was wiped and re-processed from Bronze output.")
print("✅ Bronze layer was completely untouched.")

2026-04-14 00:29:14.032 | INFO     | lakelogic.core.generator:generate:3083 - 📋 Generating data for: silver_reset_demo
2026-04-14 00:29:14.032 | INFO     | lakelogic.core.generator:generate:3084 -    Records    : 200 valid + 0 invalid = 200 total
2026-04-14 00:29:14.033 | INFO     | lakelogic.core.generator:generate:3102 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-04-14 00:29:14.038 | INFO     | lakelogic.core.generator:generate:3151 -    Row generation complete: 200 records built
2026-04-14 00:29:14.082 | INFO     | lakelogic.core.processor:run:788 - Run complete [layer=silver] | Source: 200 | Total: 200 | Good: 200 | Quarantine: 0 | Pre-Transform Dropped: 0 | Ratio: 0.00%
2026-04-14 00:29:14.082 | WARNING  | lakelogic.core.processor:run:991 - Schema drift detected for 'silver_reset_demo': missing=[], unknown=['_is_invalid', 'clean_value']
2026-04-14 00:29:14.088 | INFO     | lakelogic.core.processor:run:788 - Run complete [layer=silver] | Source: 200 | T

Run 1 (Silver): good=200, bad=0
Run 2 (Silver re-run): good=200, bad=0

✅ Silver re-processed from Bronze output. Bronze was not re-ingested.


---
## 5b. DDL-Only Mode via Pipeline (Schema Migration)

**The Problem:** You want to orchestrate table creation for your entire project before running processing.

**The Solution:** Use `pipeline.run(ddl_only=True)` to execute schema definitions to the catalog natively without processing data.

In [ ]:
# ── Generate DDL for different backends via Pipeline ─────────────────
print("── PIPELINE DDL MODE ──")
summary_ddl = pipeline.run(
    target_layers="bronze,silver",
    reset_layers="",
    reload_layers="",
    dry_run=False,
    entity_filter="",
    environment="local",
    parallel=True,
    max_workers=4,
    ddl_only=True,  # <--- DDL ONLY
    created_by="developer_experience_demo",
    reprocess_from=None,
    reprocess_to=None,
    reprocess_column=None,
    reprocess_values=None,
    retry_attempts=3,
    retry_base_wait_seconds=2,
    entity_timeout_minutes=60,
    max_consecutive_failures=2,
)

print(f"\n✅ Schema migration completed for {len(summary_ddl.results)} contracts.")
print("✅ DDL executed natively on the catalog without processing any data.")

---
## 6. Multi-Channel Alerts — Slack, Teams, Email, Webhooks

**The Problem:** A quality breach fires at 2am. Nobody sees the email until 9am. Seven hours of bad data in production.

**The Solution:** LakeLogic’s `proc.notify()` dispatches alerts to Slack, Teams, email, or any webhook — powered by Apprise with Jinja2 template support. The notification config lives in the contract.

### Real Slack Alerts from LakeLogic

| SLO Breach Alert | Quarantine Alert |
|:---:|:---:|
| ![SLO Breach](assets/slack_slo_alert.png) | ![Quarantine Alert](assets/slack_quarantine_alert.png) |

> **These are real notifications** fired by LakeLogic during a pipeline run. The contract defines which channels receive which event types.

In [8]:
# ── Contract with notification config ────────────────────────────────
# In production, these URLs would be real webhook endpoints.
# For this demo, we show what the notification system produces.

alert_yaml = """
version: 1.0.0
dataset: alert_demo

ownership:
  contacts:
    - name: Oncall Engineer
      role: owner
      email: oncall@company.com

model:
  fields:
    - name: metric_id
      type: integer
      required: true
    - name: value
      type: float

quality:
  row_rules:
    - name: positive_value
      sql: "value > 0"

quarantine:
  enabled: true
  notifications:
    - type: slack
      target: https://hooks.slack.com/services/T00/B00/demo
      on_events: [failure, slo_breach, quarantine]
    - type: teams
      target: https://outlook.webhook.office.com/demo
      on_events: [failure]
    - type: webhook
      target: https://api.pagerduty.com/v2/enqueue
      on_events: [slo_breach]
"""

print("\u2500" * 60)
print("NOTIFICATION CONFIG (from contract)")
print("\u2500" * 60)
print(alert_yaml.strip())

# ── Simulate what the notification payload looks like ───────────────
import json
from datetime import datetime, timezone

payload = {
    "event": "dataset_quality_check",
    "subject": "[LOCAL] demo/ecommerce: Dataset Quality Check Alert",
    "message": "alert_demo: quarantine rate 12% exceeds SLO threshold of 5%",
    "run_id": "abc-1234-def-5678",
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "contract": "alert_demo v1.0.0",
    "engine": "polars",
    "channels": ["slack", "teams", "pagerduty"],
}

print(f"\n{'\u2500' * 60}")
print("NOTIFICATION PAYLOAD (what Slack/Teams/Webhooks receive)")
print("\u2500" * 60)
print(json.dumps(payload, indent=2))

print("\n\u2705 One contract config → Slack + Teams + PagerDuty simultaneously.")
print("   In production, proc.notify('failure', 'message') dispatches to all channels.")

────────────────────────────────────────────────────────────
NOTIFICATION CONFIG (from contract)
────────────────────────────────────────────────────────────
version: 1.0.0
dataset: alert_demo

ownership:
  contacts:
    - name: Oncall Engineer
      role: owner
      email: oncall@company.com

model:
  fields:
    - name: metric_id
      type: integer
      required: true
    - name: value
      type: float

quality:
  row_rules:
    - name: positive_value
      sql: "value > 0"

quarantine:
  enabled: true
  notifications:
    - type: slack
      target: https://hooks.slack.com/services/T00/B00/demo
      on_events: [failure, slo_breach, quarantine]
    - type: teams
      target: https://outlook.webhook.office.com/demo
      on_events: [failure]
    - type: webhook
      target: https://api.pagerduty.com/v2/enqueue
      on_events: [slo_breach]

────────────────────────────────────────────────────────────
NOTIFICATION PAYLOAD (what Slack/Teams/Webhooks receive)
─────────────────────

## What You Just Saw

- **Structured diagnostics** — `loguru` powers every log line with timestamps, severity, and function paths
- **DDL generation** — `proc.generate_ddl()` creates CREATE TABLE DDL for any backend from the contract
- **DAG viewer** — `pipeline.visualize_dag()` renders the full dependency graph from `_system.yaml`
- **Dry run mode** — `pipeline.run(dry_run=True)` previews execution without processing or writing
- **Surgical reset** — reload Silver without re-ingesting Bronze
- **Multi-channel alerts** — Slack, Teams, email, webhooks from one contract config block

---
## Go Deeper — Explore by Capability

Each notebook below maps to a pillar of LakeLogic's [Technical Capabilities](https://lakelogic.github.io/LakeLogic/#technical-capabilities):

| # | Notebook | What You'll See |
|---|---|---|
| 🛡️ | **[Data Quality & Trust](01_data_quality_trust.ipynb)** | Reconciliation proofs, Pydantic validation, SQL-first rules, SLO monitoring |
| 📜 | **[Compliance & Governance](02_compliance_governance.ipynb)** | GDPR erasure in 2 lines, automatic lineage, cost intelligence |
| ⚡ | **[Engine & Scale](03_engine_scale.ipynb)** | Same contract on Polars & DuckDB, incremental processing, backfill |
| 🔧 | **[Developer Experience](04_developer_experience.ipynb)** | Diagnostics, DDL, DAG viewer, dry run, surgical resets, alerts |
| 🧬 | **[Data Generation & AI](05_data_generation_ai.ipynb)** | Synthetic data, referential integrity, edge case injection, contract inference |
| 🔌 | **[Integrations](06_integrations.ipynb)** | dbt adapter, dlt sources, contract-driven quality gates on arrival |

> **Each notebook is self-contained** — pick the capability that matters most to you and run it independently.